## scPRINT ingestion

```bash
uv venv .scprint --python 3.11
source .scprint/bin/activate

uv pip install scprint ipykernel "napistu-torch>=0.3.8"
python -m ipykernel install --user --name=scPRINT

# Initialize lamin database for gene annotations (optional but recommended)
lamin init --storage data/lamin_db --name scPRINT_lamin --modules bionty
```

In [1]:
import logging
import os

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from napistu.constants import ONTOLOGIES

from etl_utils import (
    SCPRINT_DEFS,
    populate_lamin_db,
    process_scprint,
)
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.constants import (
    FM_DEFS,
    FOUNDATION_MODEL_NAMES,
)
import numpy as np

logger = logging.getLogger(__name__)

INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"

SCPRINT_MODEL_PATH = os.path.join(DATA_DIR, FOUNDATION_MODEL_NAMES.SCPRINT)
os.makedirs(SCPRINT_MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
SCPRINT_VERSION_KEYS = list(SCPRINT_DEFS.VERSIONS.__dict__.keys())

populate_lamin_db()

for version in SCPRINT_VERSION_KEYS:
    process_scprint(version, OUTPUT_DIR, SCPRINT_MODEL_PATH)

→ connected lamindb: anonymous/scPRINT_lamin


INFO:etl_utils:Lamin database already configured


No module named 'triton'
FlashAttention is not installed, not using it..


INFO:etl_utils:Extracting: scPRINT small (SMALL)
INFO:etl_utils:
1. Downloading/loading model if needed...
INFO:etl_utils:Loading scPRINT model
INFO:etl_utils:Loading scPRINT model
INFO:etl_utils:Loading gene annotations


RuntimeError caught: scPrint is not attached to a `Trainer`.


INFO:etl_utils:Formatting model metadata
INFO:etl_utils:   44756 genes, 4 layers
INFO:etl_utils:2. Extracting weights...
INFO:etl_utils:   Embeddings: (44756, 128)
INFO:etl_utils:   Attention weights: 4 layers × 4 matrices (Q,K,V,O)
INFO:etl_utils:Creating FoundationModel and saving...
INFO:napistu_torch.load.foundation_models:Saving weights to output/scPRINT_small_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/scPRINT_small_metadata.json
INFO:napistu_torch.load.foundation_models:Successfully saved all results
INFO:etl_utils:Successfully saved all results!
INFO:etl_utils:Extracting: scPRINT medium (MEDIUM)
INFO:etl_utils:
1. Downloading/loading model if needed...
INFO:etl_utils:Loading scPRINT model
INFO:etl_utils:Loading scPRINT model
INFO:etl_utils:Loading gene annotations


RuntimeError caught: scPrint is not attached to a `Trainer`.


INFO:etl_utils:Formatting model metadata
INFO:etl_utils:   44756 genes, 8 layers
INFO:etl_utils:2. Extracting weights...
INFO:etl_utils:   Embeddings: (44756, 256)
INFO:etl_utils:   Attention weights: 8 layers × 4 matrices (Q,K,V,O)
INFO:etl_utils:Creating FoundationModel and saving...
INFO:napistu_torch.load.foundation_models:Saving weights to output/scPRINT_medium_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/scPRINT_medium_metadata.json
INFO:napistu_torch.load.foundation_models:Successfully saved all results
INFO:etl_utils:Successfully saved all results!
INFO:etl_utils:Extracting: scPRINT large (LARGE)
INFO:etl_utils:
1. Downloading/loading model if needed...
INFO:etl_utils:Loading scPRINT model
INFO:etl_utils:Loading scPRINT model
INFO:etl_utils:Loading gene annotations


RuntimeError caught: scPrint is not attached to a `Trainer`.


INFO:etl_utils:Formatting model metadata
INFO:etl_utils:   44756 genes, 16 layers
INFO:etl_utils:2. Extracting weights...
INFO:etl_utils:   Embeddings: (44756, 512)
INFO:etl_utils:   Attention weights: 16 layers × 4 matrices (Q,K,V,O)
INFO:etl_utils:Creating FoundationModel and saving...
INFO:napistu_torch.load.foundation_models:Saving weights to output/scPRINT_large_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/scPRINT_large_metadata.json
INFO:napistu_torch.load.foundation_models:Successfully saved all results
INFO:etl_utils:Successfully saved all results!


In [4]:
# Load results for a specific version (using MEDIUM as an example)
medium_version_id = SCPRINT_DEFS.VERSIONS.MEDIUM
file_prefix = f"{FOUNDATION_MODEL_NAMES.SCPRINT}_{medium_version_id}"
model = FoundationModel.load(OUTPUT_DIR, file_prefix)

GENES_OF_INTEREST = model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model.ordered_vocabulary]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_attn = model.weights.compute_attention_from_weights(
    layer_idx=5,
    n_heads=model.n_heads,
    vocab_mask=np.array(GENE_MASK)
)

INFO:napistu_torch.load.foundation_models:Loading weights (scPRINT_medium_weights.npz) and metadata (  scPRINT_medium_metadata.json) from output_dir (output)
INFO:napistu_torch.load.foundation_models:Successfully loaded all results
